# The TLS801_COUNTRY table 

The `TLS801_COUNTRY` table serves as a fundamental reference table within the PATSTAT Global database, providing standardised information on states, countries, territories, and intellectual property (IP) organizations. Each entity is identified by a two-letter country or authority code, which is used consistently across PATSTAT to represent geographic and institutional information.

This table is essential for geographical analysis and data harmonisation, as it ensures that country and authority codes appearing in other PATSTAT tables can be correctly interpreted and linked to meaningful descriptive information. The codes in `TLS801_COUNTRY` are primarily based on the WIPO ST.3 standard, which defines the internationally recognised two-letter codes for countries and IP offices. In addition to sovereign states, the table also includes regional and international IP organisations, allowing users to distinguish between national and supranational patent authorities.

The `TLS801_COUNTRY` table is widely used in conjunction with other PATSTAT tables, such as those containing patent applications, publications, applicants, and inventors. By joining these tables through the country or authority code fields, users can enrich patent data with geographic context, enabling analyses such as identifying filing trends by country, mapping inventor or applicant locations, and studying the international scope of patent activity.

In [12]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS801_COUNTRY
from sqlalchemy import case, func, select, distinct

# Initialise the PATSTAT client
patstat = PatstatClient(env="TEST")

# Access ORM
db = patstat.orm()

## Identification Codes & Names

These fields provide standard references for identifying a specific entity.

### CTRY_CODE
This field contains the two-letter code representing states, territories, or intergovernmental organizations. It follows the **WIPO ST.3** standard (with minor additions).

### ISO_ALPHA3
This field contains the three-letter code representing countries or territories. It follows the **ISO 3166** standard.

### ST3_NAME
This field contains the short English name of the state, territory, or intergovernmental organization. The naming convention follows the **WIPO ST.3** standard (with minor extensions).

An example is shown on how to visualise the different countries

In [5]:
q = (
    db.query(
        TLS801_COUNTRY.ctry_code,
        TLS801_COUNTRY.iso_alpha3,
        TLS801_COUNTRY.st3_name,
    )
    .order_by(TLS801_COUNTRY.ctry_code)
)

res = patstat.df(q)
res

,ctry_code,iso_alpha3,st3_name
0,None,NAM,Namibia
1,AE,ARE,United Arab Emirates
2,AF,AFG,Afghanistan
3,AG,ATG,Antigua and Barbuda
4,AI,AIA,Anguilla
...,...,...,...
236,YE,YEM,Yemen
237,YU,YUG,Yugoslavia/Serbia and Montenegro
238,ZA,ZAF,South Africa
239,ZM,ZMB,Zambia


## Classification and Location

The following fields describe the nature of the entity represented by the country code and its geographical location.

### ORGANISATION_FLAG
This field indicates whether the `CTRY_CODE` refers to a sovereign state or territory or to an intergovernmental organisation. It allows users to distinguish country codes from codes representing organisations such as the European Patent Office (EPO) or other international IP authorities.

### CONTINENT
This field provides the English name of the continent in which the state or territory is located. The geographic classification follows definitions published on Wikipedia.


## Membership Indicators
These fields specify whether a country or territory is a member of major international and regional organisations. They are particularly useful for grouping countries and filtering data in comparative analyses.

### EU_MEMBER
Indicates whether the country or territory is a member state of the European Union (EU).

### EPO_MEMBER
Indicates whether the country or territory is a member state of the European Patent Organisation (EPO).

### OECD_MEMBER
Indicates whether the country or territory is a member state of the Organisation for Economic Co-operation and Development (OECD).

The membership and classification fields in `TLS801_COUNTRY` can be used to enrich analyses across PATSTAT by joining this table with any dataset that contains a country code.
A common example is the `TLS206_PERSON` table, which includes the `PERSON_CTRY_CODE` field identifying the country associated with each individual. By joining `TLS206_PERSON` with `TLS801_COUNTRY`, analysts can:

* Filter individuals based on membership criteria, such as selecting only persons from EPO and OECD member countries
* Compare patent-related activity inside versus outside EPO member states
* Analyse the distribution of individuals by sector (using the `PSN_SECTOR` field) across different geographic or institutional groupings

Below it is shown a practical example:

In [14]:
from epo.tipdata.patstat.database.models import TLS206_PERSON
q = (
    db.query(
        TLS206_PERSON.psn_sector,
        TLS801_COUNTRY.epo_member,
        func.count(distinct(TLS206_PERSON.person_id)).label("person_count")
    )
    .join(
        TLS801_COUNTRY, 
        TLS801_COUNTRY.ctry_code == TLS206_PERSON.person_ctry_code  # Join on country code
    )
    .group_by(
        TLS206_PERSON.psn_sector,
        TLS801_COUNTRY.epo_member,
    )
    .order_by(
        TLS801_COUNTRY.epo_member,
        TLS206_PERSON.psn_sector
    )
)

res = patstat.df(q)
res

,psn_sector,epo_member,person_count
0,None,,80014
1,COMPANY,,22962
2,COMPANY GOV NON-PROFIT,,68
3,COMPANY UNIVERSITY,,9
4,GOV NON-PROFIT,,746
5,GOV NON-PROFIT UNIVERSITY,,80
6,HOSPITAL,,3
7,INDIVIDUAL,,67236
8,UNIVERSITY,,1828
9,UNKNOWN,,14193


To express the same distribution using percentages rather than absolute counts:

In [19]:
person_count = func.count(
    distinct(TLS206_PERSON.person_id)
)

total_per_sector = func.sum(
    person_count
).over(
    partition_by=TLS206_PERSON.psn_sector
)

q = (
    db.query(
        TLS206_PERSON.psn_sector.label("psn_sector"),
        TLS801_COUNTRY.epo_member.label("epo_member"),
        (
            person_count / total_per_sector
        ).label("share")
    )
    .join(
        TLS801_COUNTRY,
        TLS801_COUNTRY.ctry_code == TLS206_PERSON.person_ctry_code
    )
    .group_by(
        TLS206_PERSON.psn_sector,
        TLS801_COUNTRY.epo_member
    )
    .order_by(
        TLS206_PERSON.psn_sector,
        TLS801_COUNTRY.epo_member
    )
)

res = patstat.df(q)
res

,psn_sector,epo_member,share
0,None,,0.64022468
1,None,Y,0.35977532
2,COMPANY,,0.633521865
3,COMPANY,Y,0.366478135
4,COMPANY GOV NON-PROFIT,,0.80952381
5,COMPANY GOV NON-PROFIT,Y,0.19047619
6,COMPANY HOSPITAL,Y,1
7,COMPANY UNIVERSITY,,0.5625
8,COMPANY UNIVERSITY,Y,0.4375
9,GOV NON-PROFIT,,0.617549669


## Other fields

### DISCONTINUED
This field indicates whether a state or organization no longer exists. It is based on WIPO standard ST.3 and can be useful for historical data analysis involving entities that have dissolved or changed names (e.g., the Soviet Union).

In [13]:
q = (
    db.query(
        TLS801_COUNTRY.ctry_code,
        TLS801_COUNTRY.st3_name,
        TLS801_COUNTRY.discontinued
    )
    .filter(
        TLS801_COUNTRY.discontinued == "Y"
    )
    .order_by(
        TLS801_COUNTRY.ctry_code
    )
)

res = patstat.df(q)
res

,ctry_code,st3_name,discontinued
0,AN,Netherlands Antilles,Y
1,BU,Burma,Y
2,CS,Czechoslovakia,Y
3,DD,German Democratic Republic,Y
4,DL,German Democratic Republic,Y
5,SU,Soviet Union,Y
6,YD,Democratic Yemen,Y
7,YU,Yugoslavia/Serbia and Montenegro,Y
